# 01 — NeighborsMatch: accuracy vs. problem radius

**Main experiment (strong claim).** Alon & Yahav's NeighborsMatch is the canonical over-squashing probe:
vanilla GNNs collapse as the radius grows because information from radius-`r` leaves is squashed through a
single path into the root's fixed-width embedding.

We compare **GCN / GAT / GIN** against our **QuotientNet** (kQ/I-aware aggregation). The claim: the quotient
layer sustains accuracy where the baselines fall, by collapsing functionally equivalent paths *before*
aggregating.

> ⚠️ **Known WIP (see README):** batched `edge_classes` need a tiling+offset collate before full runs.
> This notebook first runs a **smoke test** (single radius, small sample) to validate the pipeline, then the
> full sweep. If the quotient layer doesn't beat baselines by the day-7 checkpoint, pivot to `03_diagnostic.ipynb`.

In [ ]:
import yaml, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from pathlib import Path
from oversquash.data import neighborsmatch_dataset
from oversquash.train import run_neighborsmatch_sweep

CFG = yaml.safe_load(open('../configs/neighborsmatch.yaml'))
RESULTS = Path('../results'); (RESULTS / 'figures').mkdir(parents=True, exist_ok=True); (RESULTS / 'tables').mkdir(parents=True, exist_ok=True)
CFG

## Smoke test (day 1–2): validate the full pipeline at one small radius

Keep it tiny so it runs in seconds. We only want to confirm: data generates, every model trains one sweep
point, and the quotient model consumes its edge classes without shape errors.

In [ ]:
smoke = dict(CFG)
smoke['data'] = dict(CFG['data'], radii=[2], n_samples=512)
smoke['train'] = dict(CFG['train'], epochs=20, batch_size=128, patience=10)
smoke_results = run_neighborsmatch_sweep(smoke, neighborsmatch_dataset)
pd.DataFrame(smoke_results)

## Full radius sweep (day 3–4)

Run after the smoke test passes and the batched-edge-class collate is in place. Expect baselines to degrade
monotonically with radius; QuotientNet should hold up longer.

In [ ]:
results = run_neighborsmatch_sweep(CFG, neighborsmatch_dataset)
df = pd.DataFrame(results)
df.to_csv(RESULTS / 'tables' / 'neighborsmatch.csv', index=False)
df.pivot(index='radius', columns='model', values='val_acc')

In [ ]:
# Figure: accuracy vs radius, one line per model (the paper's headline figure).
fig, ax = plt.subplots(figsize=(6, 4))
for name, g in df.groupby('model'):
    g = g.sort_values('radius')
    ax.plot(g['radius'], g['val_acc'], marker='o', label=name)
ax.set_xlabel('problem radius $r$'); ax.set_ylabel('accuracy')
ax.set_title('NeighborsMatch: kQ/I vs. baselines')
ax.axhline(0.0, color='grey', lw=0.5); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RESULTS / 'figures' / 'neighborsmatch_accuracy_vs_radius.pdf')
fig.savefig(RESULTS / 'figures' / 'neighborsmatch_accuracy_vs_radius.png', dpi=150)
plt.show()